## Import libraries

In [1]:
import llm_funcs.funcs as llm_helpers

from transformers import DPRContextEncoder, DPRContextEncoderTokenizer, DPRQuestionEncoder, DPRQuestionEncoderTokenizer, AutoTokenizer, AutoModelForCausalLM

import random
import faiss

/Users/manfernandez/anaconda3/envs/ai_agents/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Read text file used as context

In [2]:
paragraphs = llm_helpers.read_and_split_text('source_files/companyPolicies.txt')

random.seed(42)
random.shuffle(paragraphs)

## Process context

LLMs work by tokenizing and embedding the given text into vector representations that can later be searched for relevant queried information. In this project, I'm using a Dense Passage Retriever (DPR) model to encode passages. The employed tokenizers and encoders come from the HuggingFace library.

In [3]:
# Define context parameters
MAX_LEN = 256 # Max numnber of tokens

In [4]:
# Context tokenization
ctxt_tokenizer = DPRContextEncoderTokenizer.from_pretrained('facebook/dpr-ctx_encoder-single-nq-base')

# Context embedding
ctxt_encoder = DPRContextEncoder.from_pretrained('facebook/dpr-ctx_encoder-single-nq-base')

ctxt_embeddings = llm_helpers.encode_contexts(paragraphs, ctxt_tokenizer, ctxt_encoder, MAX_LEN)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 35780.45it/s]
[transformers] DPRContextEncoder LOAD REPORT from: facebook/dpr-ctx_encoder-single-nq-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
ctx_encoder.bert_model.pooler.dense.bias   | UNEXPECTED |  | 
ctx_encoder.bert_model.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


I'm using the Facebook AI Similarity Search (FAISS) to identify index embedded contexts and efficiently search them for similarities to an user's query.

In [5]:
# Define FAISS parameters
EMBEDDING_DIMS = 768 # Matches dimenions of embeddings defined by DPR model

In [6]:
# Create FAISS index for the embeddings - consider converting embeddings to float32
index = faiss.IndexFlatL2(EMBEDDING_DIMS)
index.add(ctxt_embeddings)

## Process queries

The same process of tokenization and embedding is performed on the user's queries. In this case, the same models are trained differently to optimize question processing

In [7]:
# Question tokenization
qstn_tokenizer = DPRQuestionEncoderTokenizer.from_pretrained('facebook/dpr-question_encoder-single-nq-base')

# Question embedding
qstn_encoder = DPRQuestionEncoder.from_pretrained('facebook/dpr-question_encoder-single-nq-base')

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 52929.21it/s]
[transformers] DPRQuestionEncoder LOAD REPORT from: facebook/dpr-question_encoder-single-nq-base
Key                                             | Status     |  | 
------------------------------------------------+------------+--+-
question_encoder.bert_model.pooler.dense.bias   | UNEXPECTED |  | 
question_encoder.bert_model.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Generate responses using LLMs

After retrieval, the LLM combines the RAG-derived information with its knowledge to generate text to produce full, coherent answers.

In [8]:
llm_tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
llm_model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2")
#llm_model.generation_config.pad_token_id = llm_tokenizer.pad_token_id
llm_model.generation_config.pad_token_id = llm_tokenizer.eos_token_id # Try to see if it improves


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 10566.98it/s]


## Try RAG-assisted LLM for user's questions

In [9]:
# Define user query
question = 'What is mobile policy?'

# Retrieve relevant passages from context
D, I = llm_helpers.search_relevant_contexts(question, qstn_tokenizer,
                                            qstn_encoder, index, k=5)
top_contexts = [paragraphs[idx] for idx in I[0]]

I want to test different parameters to fine-tune how the LLM uses RAG  to retrieve context and generate human-readable answers

In [10]:
# Settings correspond to the maximum number of new tokens generated,
# the minimum length, the length penalty and the number of beams explored in parallel

settings = [
    (50, 50, 1.0, 2),
    (120, 30, 2.0, 4),
    (100, 20, 2.5, 6)
]

for setting in settings:
    answer = llm_helpers.generate_answer(llm_tokenizer, llm_model, question, top_contexts, *setting)
    print(f"Settings: max_new_tokens={setting[0]}, min_length={setting[1]}, length_penalty={setting[2]}, num_beams={setting[3]}")
    print("Generated Answer:", answer)
    print("\n" + "="*80 + "\n")
    
    

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Settings: max_new_tokens=50, min_length=50, length_penalty=1.0, num_beams=2
Generated Answer: What is mobile policy? 4.	Mobile Phone Policy The Mobile Phone Policy sets forth the standards and expectations governing the appropriate and responsible usage of mobile devices in the organization. The purpose of this policy is to ensure that employees utilize mobile phones in a manner consistent with company values and legal compliance. Monitoring: The company retains the right to monitor internet and email usage for security and compliance purposes. Acceptable Use: Mobile devices are primarily intended for work-related tasks. Limited personal usage is allowed, provided it does not disrupt work obligations. The Mobile Phone Policy is aimed at promoting the responsible and secure use of mobile devices in line with legal and ethical standards. Every employee is expected to comprehend and abide by these guidelines. Regular reviews of the policy ensure its ongoing alignment with evolving technol